In [1]:
import json
import pandas as pd
from itertools import chain
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from datetime import datetime
import re
from functools import reduce
from operator import mul
import requests
from digital_assistant_first.aviasales_system.aviasales_travelpayouts_helper import ProvidersIATAMatcher
from digital_assistant_first.aviasales_system.aviasales_recommendation_engine import ru_months, pass_mapping_0, pass_mapping_1, format_passengers, cities_mapping, airlines_mapping, airports_mapping
from typing import Dict, List

airline_matcher = ProvidersIATAMatcher()

class AviasalesIATAConverterCity:
    def __init__(self):
        self.query = "https://suggest.aviasales.com/v2/places.json?locale=ru_RU&max=7&term={}&types[]=city&types[]=airport&types[]=country"

    def get_cities(self, flight_info_dict: Dict[str, str]) -> List[str]:
        origin_res_raw = requests.get(
            self.query.format(flight_info_dict["origin_airport_iata"])
        ).json()
        destination_res_raw = requests.get(
            self.query.format(flight_info_dict["destination_airport_iata"])
        ).json()
        if origin_res_raw:
            if origin_res_raw[0]["type"] == "airport":
                origin_res = origin_res_raw[0]["city_name"]
            if origin_res_raw[0]["type"] == "city":
                origin_res = origin_res_raw[0]["name"]
        else:
            origin_res = None
        if destination_res_raw:
            if destination_res_raw[0]["type"] == "airport":
                destination_res = destination_res_raw[0]["city_name"]
            if destination_res_raw[0]["type"] == "city":
                destination_res = destination_res_raw[0]["name"]
        else:
            destination_res = None
        
        return origin_res, destination_res
    
iata_city_converter = AviasalesIATAConverterCity()

In [2]:
with open("parsed_tickets_data_airscraper.json", "r") as f:
    data = json.load(f)

In [3]:
def format_date_russian_full(date_str):
    date_obj, time_obj = date_str.split()
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    time_obj = datetime.strptime(time_obj, "%H:%M:%S")
    
    day = date_obj.day
    month = ru_months[date_obj.month]
    year = date_obj.year

    hr = time_obj.hour
    min = time_obj.minute
    
    return f"{day} {month}", f"{hr}:{min}"

In [4]:
smpl = data[1]

In [5]:
def get_airlines_naming(leg_dict):
    try:
        res_dict = leg_dict.copy()
        res_dict["airline"] = airlines_mapping.loc[airline_matcher.match_iata(res_dict["airline"])]["name"]
        return res_dict
    except Exception:
        return leg_dict

get_airlines_naming(smpl["legs"][0])

{'origin_city': 'Baku',
 'origin_airport_name': 'Baku Heydar Aliyev International',
 'origin_airport_iata': 'GYD',
 'destination_city': 'Dubai',
 'destination_airport_name': 'Dubai',
 'destination_airport_iata': 'DXB',
 'duration': 175,
 'departure_ts': '2025-05-22 18:00:00',
 'arrival_ts': '2025-05-22 20:55:00',
 'airline': 'Emirates'}

In [6]:
def get_russian_airport_naming(leg_dict):
    res_dict = leg_dict.copy()
    for k, v in leg_dict.items():
        try:
            if k == "origin_airport_name":
                res_dict[k] = airports_mapping.loc[leg_dict["origin_airport_iata"]]["name"]
            elif k == "destination_airport_name":
                res_dict[k] = airports_mapping.loc[leg_dict["destination_airport_iata"]]["name"]
        except Exception:
            continue
    return res_dict
get_russian_airport_naming(smpl["legs"][0])

{'origin_city': 'Baku',
 'origin_airport_name': 'Гейдар Алиев',
 'origin_airport_iata': 'GYD',
 'destination_city': 'Dubai',
 'destination_airport_name': 'Дубай',
 'destination_airport_iata': 'DXB',
 'duration': 175,
 'departure_ts': '2025-05-22 18:00:00',
 'arrival_ts': '2025-05-22 20:55:00',
 'airline': 'Emirates'}

In [7]:
def get_russian_city_naming(leg_dict):
    try:
        res_dict = leg_dict.copy()
        orig, dest = iata_city_converter.get_cities(leg_dict)
        res_dict["origin_city"] = orig
        res_dict["destination_city"] = dest
        return res_dict
    except Exception:
        return leg_dict
get_russian_city_naming(smpl["legs"][0])

{'origin_city': 'Баку',
 'origin_airport_name': 'Baku Heydar Aliyev International',
 'origin_airport_iata': 'GYD',
 'destination_city': 'Дубай',
 'destination_airport_name': 'Dubai',
 'destination_airport_iata': 'DXB',
 'duration': 175,
 'departure_ts': '2025-05-22 18:00:00',
 'arrival_ts': '2025-05-22 20:55:00',
 'airline': 'Emirates'}

In [8]:
smpl["legs"] = [get_airlines_naming(get_russian_city_naming(get_russian_airport_naming(leg))) for leg in smpl["legs"]]

In [9]:
smpl

{'price': 2425.94,
 'general_departure_ts': '2025-05-22 18:00:00',
 'general_arrival_ts': '2025-05-23 16:50:00',
 'legs': [{'origin_city': 'Баку',
   'origin_airport_name': 'Гейдар Алиев',
   'origin_airport_iata': 'GYD',
   'destination_city': 'Дубай',
   'destination_airport_name': 'Дубай',
   'destination_airport_iata': 'DXB',
   'duration': 175,
   'departure_ts': '2025-05-22 18:00:00',
   'arrival_ts': '2025-05-22 20:55:00',
   'airline': 'Emirates'},
  {'origin_city': 'Дубай',
   'origin_airport_name': 'Дубай',
   'origin_airport_iata': 'DXB',
   'destination_city': 'Маэ',
   'destination_airport_name': 'Сейшелы',
   'destination_airport_iata': 'SEZ',
   'duration': 280,
   'departure_ts': '2025-05-23 08:55:00',
   'arrival_ts': '2025-05-23 13:35:00',
   'airline': 'Emirates'},
  {'origin_city': 'Маэ',
   'origin_airport_name': 'Сейшелы',
   'origin_airport_iata': 'SEZ',
   'destination_city': 'Антананариву',
   'destination_airport_name': 'Антананариву',
   'destination_airport_

In [ ]:
for 

In [4]:
iata_airports

[{'name_translations': {'en': 'Erave'},
  'city_code': 'ERE',
  'country_code': 'PG',
  'time_zone': 'Pacific/Port_Moresby',
  'code': 'ERE',
  'iata_type': 'airport',
  'name': 'Эрейв',
  'coordinates': {'lat': -6.633333, 'lon': 143.9},
  'flightable': False},
 {'name_translations': {'en': 'Mouakchott'},
  'city_code': 'ATR',
  'country_code': 'MR',
  'time_zone': 'Africa/Nouakchott',
  'code': 'ATR',
  'iata_type': 'airport',
  'name': 'Атар',
  'coordinates': {'lat': 20.499443, 'lon': -13.046389},
  'flightable': False},
 {'name_translations': {'en': 'Hang Nadim International Airport'},
  'city_code': 'BTH',
  'country_code': 'ID',
  'time_zone': 'Asia/Jakarta',
  'code': 'BTH',
  'iata_type': 'airport',
  'name': 'Ханг Надим',
  'coordinates': {'lat': 1.123627, 'lon': 104.11528},
  'flightable': True},
 {'name_translations': {'en': 'San Felipe'},
  'city_code': 'SNF',
  'country_code': 'VE',
  'time_zone': 'America/Caracas',
  'code': 'SNF',
  'iata_type': 'airport',
  'name': 'Сан